In [ ]:
import pandas as pd
from pandasql import sqldf
import numpy as np
import csv
import os

FEB="Z:/CARTELLA rete GAF/► NEW/Car Fleet & Mobility/Fuel/Fuel Violation/2026/2. february/fattura_PJ11164801_28022026.xlsx"
fleet_2026="Z:/CARTELLA rete GAF/► NEW/Car Fleet & Mobility/Car Fleet/Car Fleet 2026.xlsx"


df_FEB=pd.read_excel(FEB)

df_FEB['Date']=pd.to_datetime(df_FEB['Date'],errors='coerce',dayfirst=True).dt.strftime(f"%d/%m/%y")


df_car_fleet_FEB=pd.read_excel(fleet_2026,sheet_name="Car Fleet 2026")

n_colonna=[]
for i in df_car_fleet_FEB["Model"]:
    if i is np.nan:
        n_colonna.append(np.nan)
    else:
        n_colonna.append(i.split(" ")[0])
df_car_fleet_FEB["Marchio"]=n_colonna

n_colonna.clear()

#sacar soo el modello
for i in df_car_fleet_FEB["Model"]:
    if i is np.nan:
        n_colonna.append(np.nan)
    else:
        n_colonna.append(" ".join(i.split(" ")[1:]))
df_car_fleet_FEB["Modello"]=n_colonna



df_car_fleet_FEB=df_car_fleet_FEB[["Plate","Model","User","Div","Modello","Marchio"]]






In [49]:
#rifornimenti e costi
#chi fa più rifornimenti benzina (persona inviduale)

query1="""
        SELECT User,Sum([Full amount]) AS "Costi totali" ,Count('Ticket No.') as "Rifornimenti",Div
        FROM df_FEB FEB
        JOIN df_car_fleet_FEB cff 
        ON FEB.[Plate number] = cff.Plate
        GROUP BY User
        ORDER BY Rifornimenti DESC
        LIMIT 10
    """



#chi fa più rifornimenti benzina (dipartimento)
query2="""
        SELECT Div,Sum([Full amount]) AS "Costi totali",Count('Ticket No.') as "Rifornimenti"
        FROM df_FEB FEB
        JOIN df_car_fleet_FEB cff 
        ON FEB.[Plate number] = cff.Plate
        GROUP BY Div       
        ORDER BY "Costi totali" DESC
        Limit 10
    """





#top costi di AS
query3="""
        SELECT User,Sum([Full amount]) AS "Costi totali" ,Count('Ticket No.') as "Rifornimenti" 
        FROM df_FEB FEB
        JOIN df_car_fleet_FEB cff 
        ON FEB.[Plate number] = cff.Plate
        WHERE Div = "AS"
        GROUP BY User
        ORDER BY "Costi totali" DESC
        LIMIT 10
    """





#query 1 - 3
df_service_type=sqldf(query3,globals())
print(df_service_type)

                 User  Costi totali  Rifornimenti
0     BAZZURRI MATTIA        574.82            11
1    DI ROCCO ROBERTO        478.05             7
2     SABATTI STEFANO        451.10             7
3        SABA ROBERTO        428.64             5
4            POOL CAR        428.64             5
5   CIUCCI ALESSANDRO        428.64             5
6  CHIARAMONTE DAVIDE        428.64             5
7   DONZELLI RICCARDO        398.29             6
8           RIVA LUCA        373.79             5
9       FERLITO SANTO        353.03             6


In [ ]:
#quali zone sono più transitate e/o city rifornimento,               


#preferenza città di ogni divisione
query1="""
        SELECT Div,Citta as "Citta' più transitata"
        FROM (
                SELECT Div , City as "Citta" ,COUNT(*) as "Totale", ROW_NUMBER()OVER(PARTITION BY Div ORDER BY COUNT (*) DESC) as top
                FROM df_FEB FEB
                JOIN df_car_fleet_FEB cff 
                ON FEB.[Plate number] = cff.Plate
                GROUP BY Div, City
                )
        WHERE top = 1
"""



#Rank di citta' più frequentata per Div 
query2="""
SELECT Preferenze as "Città", SUM("Citta' popolare") as "Div"
FROM (
        SELECT *,
        CASE 
                WHEN "Rank" = 3 THEN "Altri"  
                ELSE "Citta'"
        END as "Preferenze"
        FROM(
                SELECT "Citta'",Rank() Over (ORDER BY "Citta' popolare" DESC) as "Rank", "Citta' popolare"
                FROM(
                        SELECT "Citta' più transitata" as "Citta'",Count("Citta' più transitata") as "Citta' popolare"
                        FROM(
                                SELECT Div,Citta as "Citta' più transitata"
                                FROM (
                                        SELECT Div , City as "Citta" ,COUNT(*) as "Totale", ROW_NUMBER()OVER(PARTITION BY Div ORDER BY COUNT (*) DESC) as top
                                        FROM df_FEB FEB
                                        JOIN df_car_fleet_FEB cff 
                                        ON FEB.[Plate number] = cff.Plate
                                        GROUP BY Div, City
                                        )
                                WHERE top = 1
                                )
                        GROUP BY "Citta'"
                        ORDER BY "Citta' popolare" DESC
                        )
                )
        )
GROUP BY Preferenze
ORDER BY CASE
        WHEN "Città" = "Altri" THEN 3
END
"""



#GROUP BY Preferenze
#ORDER BY CASE                                      metti l'ordine che vuoi 1,2,3
#        WHEN "Città" = "Altri" THEN 3
#        WHEN "Città" = "wasa" THEN 1
#        WHEN "Città" = "nose" THEN 2
#END




df_service_type=sqldf(query2,globals())
print(df_service_type)

                   Città  Div
0                 MILANO   19
1  SAN GIULIANO MILANESE    2
2                  Altri    7


In [ ]:
#Ci sono più macchine a gasolio o senza pb     per persona           
query1="""
    SELECT "Prod.", Count(DISTINCT([Plate number])) as "Macchine" 
    FROM df_FEB FEB
    JOIN df_car_fleet_FEB cff 
    ON FEB.[Plate number] = cff.Plate
    GROUP BY "Prod."
    ORDER BY "Macchine" DESC
"""

#Ci sono più macchine a gasolio o senza pb     per Div
query2="""
SELECT "Prod." , COUNT(*) AS "Preferenze_DIV"
FROM(
    SELECT Div,"Prod."
    FROM(
        SELECT Div, ROW_NUMBER()OVER(PARTITION BY Div ORDER BY COUNT (*) DESC) as top ,"Prod."
        FROM df_FEB FEB
        JOIN df_car_fleet_FEB cff 
        ON FEB.[Plate number] = cff.Plate
        GROUP BY Div
        )
    WHERE top = 1
    )
GROUP BY "Prod."
ORDER BY "Preferenze_DIV" DESC
"""


#quale viene usata di piu (gasolio o senza pb) per città
query3="""
SELECT City,"Prod."
FROM(
    SELECT City, ROW_NUMBER()OVER(PARTITION BY Div ORDER BY COUNT (*) DESC) as top ,"Prod."
    FROM df_FEB FEB
    JOIN df_car_fleet_FEB cff 
    ON FEB.[Plate number] = cff.Plate
    GROUP BY City
    )
WHERE top = 1
"""





df_service_type=sqldf(query2,globals())
print(df_service_type)

            Prod.  Preferenza Div
0         GASOLIO              21
1  SUPER SENZA PB               7


In [62]:
#quale card No. (carta di credito) viene usata di piu


#Chi fa più rifornimenti
query1="""
SELECT "Card No.", USER , COUNT(*) as "Rifornimenti",SUM("Full amount") as "Totale costo"
FROM df_FEB FEB
JOIN df_car_fleet_FEB cff 
ON FEB.[Plate number] = cff.Plate
GROUP BY "Card No.", USER
ORDER BY "Rifornimenti" DESC
LIMIT 10
"""

#Quale carta viene usata di più, e costo totale
query2="""
SELECT "Card No.", COUNT(*) as "Rifornimenti", SUM("Full amount") as "Totale costo" ,Modello, Marchio
FROM df_FEB FEB
JOIN df_car_fleet_FEB cff 
ON FEB.[Plate number] = cff.Plate
GROUP BY "Card No."
ORDER BY "Rifornimenti" DESC
LIMIT 10
"""

df_service_type=sqldf(query2,globals())
print(df_service_type)

   Card No.  Rifornimenti  Totale costo  \
0       396            20       1714.56   
1       362            16       1054.16   
2       444            12        717.90   
3       443            12        808.47   
4       386            12        589.23   
5       398            11        574.82   
6       367             8        625.18   
7       350             8        626.32   
8       346             8        302.54   
9       407             7        451.10   

                                           Modello     Marchio  
0  Variant 2.0TDI SCR EVO 110kW Business DSG 150cv      Passat  
1                 8 Variant 2.0 TDI Life DSG 115cv        Golf  
2     Focus 1.5 Ecoblue 115cv ST-Line Auto Euro 6D        Ford  
3        Tiguan 2.0 Tdi Scr 110kw Edition Plus DSG  VOLKSWAGEN  
4                Golf Variant 2.0 TDI SCR DSG Life  VOLKSWAGEN  
5            Stonic 1.0 TGDI 120 CV MHEV DCT Style         Kia  
6  Variant 2.0TDI SCR EVO 110kW Business DSG 150cv      Passat  
7     

In [65]:
#quale macchina viene usata da più persone, manca il marchio
query1="""
SELECT Marchio, Preferenze, Modello
FROM (
    SELECT [Plate number], COUNT(DISTINCT User) as "Preferenze", Marchio , Modello
    FROM df_FEB FEB
    JOIN df_car_fleet_FEB cff 
    ON FEB.[Plate number] = cff.Plate
    GROUP BY [Plate number]
    ORDER BY "Preferenze" DESC
    LIMIT 10
    )
"""

df_service_type=sqldf(query1,globals())
print(df_service_type)

      Marchio  Preferenze                                          Modello
0      Passat           4  Variant 2.0TDI SCR EVO 110kW Business DSG 150cv
1        Golf           4                 8 Variant 2.0 TDI Life DSG 115cv
2        Ford           3     Focus 1.5 Ecoblue 115cv ST-Line Auto Euro 6D
3  VOLKSWAGEN           3        Tiguan 2.0 Tdi Scr 110kw Edition Plus DSG
4  VOLKSWAGEN           3                Golf Variant 2.0 TDI SCR DSG Life
5  VOLKSWAGEN           3                Golf Variant 2.0 TDI SCR DSG Life
6        Audi           3             A3 2020 SPB 35 TDI S tronic Business
7      Passat           2  Variant 2.0TDI SCR EVO 110kW Business DSG 150cv
8     Renault           2                Clio 1.5 DCI Blue 74 Kw Evolution
9        Ford           2    Focus 1.5 EcoBlue 120cv DSG SW Business Euro6


In [66]:
#maggiormente dove fanno più rifornimenti?   'Point of sale type'
query1="""
SELECT "Point of sale type", COUNT(*) as "Preferenza"
FROM df_FEB FEB
JOIN df_car_fleet_FEB cff 
ON FEB.[Plate number] = cff.Plate
GROUP BY "Point of sale type"
ORDER BY "Point of sale type" DESC
"""


df_service_type=sqldf(query1,globals())
print(df_service_type)

  Point of sale type  Preferenza
0          Ordinario         241
1               Easy          53
2       Autostradale          75


In [ ]:
#informnazioni generali su ogni marchio macchina
#Quale marchio di macchina usano viene usata da più persone
query1="""
SELECT Rank() Over (ORDER BY "Users" DESC) as "Rank",*
FROM (
    SELECT "Marchio_da_usare" as "Marchio" , COUNT(DISTINCT("User")) AS "Users"
    FROM(
        SELECT *,
        CASE 
            WHEN "Marchio" = "Audi" THEN "AUDI"
            WHEN "Marchio" = "Bmw" THEN "BMW"
            WHEN "Marchio" = "Ford" THEN "FORD"
            WHEN "Marchio" = "kia" THEN "KIA"
            WHEN "Marchio" = "Kia" THEN "KIA"
                ELSE "Marchio"
            END as "Marchio_da_usare"
        FROM df_FEB FEB
        JOIN df_car_fleet_FEB cff 
        ON FEB.[Plate number] = cff.Plate
        )
    GROUP BY "Marchio_da_usare"
    ORDER BY "Users" DESC
    )
"""

#Quale marchio di macchina usano viene usata o preferita da più div
query2="""
SELECT Rank() Over (ORDER BY "Preferenza" DESC) as "Rank",*
FROM(
    SELECT Marchio , COUNT(Marchio)  as "Preferenza"
    FROM (
        SELECT Div, Marchio
        FROM(
            SELECT Div, ROW_NUMBER()OVER(PARTITION BY Div ORDER BY COUNT(DISTINCT([Plate number])) DESC) as top, Marchio
            FROM df_FEB FEB
            JOIN df_car_fleet_FEB cff 
            ON FEB.[Plate number] = cff.Plate
            GROUP BY Div
            )
        WHERE top = 1
    )
    GROUP BY Marchio
    ORDER BY "Preferenza" DESC
)
"""

#QUALI SONO I DIV CHE PREFERISCONO IL VOLKSWAGEN
query3="""
SELECT *
FROM(
    SELECT Div, Marchio
    FROM(
        SELECT Div, ROW_NUMBER()OVER(PARTITION BY Div ORDER BY COUNT(DISTINCT([Plate number])) DESC) as top, Marchio
        FROM df_FEB FEB
        JOIN df_car_fleet_FEB cff 
        ON FEB.[Plate number] = cff.Plate
        GROUP BY Div
        )
    WHERE top = 1
)
WHERE Marchio = "VOLKSWAGEN"
"""

#Quale modello VOLKSWAGEN viene usata da più perosne (ranking)
query4="""
SELECT Rank() Over (ORDER BY "Preferenze" DESC) as "Rank",*
FROM (
    SELECT Modello, COUNT(DISTINCT(User)) AS "Preferenze"
    FROM df_FEB FEB
    JOIN df_car_fleet_FEB cff 
    ON FEB.[Plate number] = cff.Plate
    WHERE Marchio = "VOLKSWAGEN"
    GROUP BY Modello
    ORDER BY "Preferenze" DESC
)
"""

df_service_type=sqldf(query4,globals())
print(df_service_type)

In [ ]:
#[Phyton q4]     


#####quale posto di benzina costa di meno in media a secondo la città

df_FEB_da_analizzare=df_FEB
df_FEB_da_analizzare["Date"]=pd.to_datetime(df_FEB_da_analizzare['Date'],errors='coerce',dayfirst=True)
df_FEB_da_analizzare["Day"]=df_FEB_da_analizzare["Date"].dt.day

df_FEB_da_analizzare["quartile"]=pd.cut(
    df_FEB_da_analizzare["Day"],
    bins=[0,7,14,21,df_FEB_da_analizzare["Day"].max()],
    labels=["Q1","Q2","Q3","Q4"]
)

df_FEB_da_analizzare[["quartile","Prezzo EUR/l"]]    

In [ ]:
#Quale città costa di meno a secondo il prezzo, prezzo base gasolio e senza pb  ; diviso in 4 settimane       SELF SERVICE

#GASOLIO

query1="""
SELECT Settimane_al_mese, City, Settimana_minima
FROM(
    SELECT *,
    CASE 
        WHEN "Settimana" = "Q1" THEN "1"
        WHEN "Settimana" = "Q2" THEN "2"
        WHEN "Settimana" = "Q3" THEN "3"
        WHEN "Settimana" = "Q4" THEN "4"
            ELSE "Settimana"
        END AS "Settimane_al_mese"
    FROM(
        SELECT quartile as "Settimana" , City ,MIN(MEDIA_SETTIMANA) AS "Settimana_minima"
        FROM(
            SELECT City,quartile, avg("MEDIA_GIORNO") as "MEDIA_SETTIMANA"
            FROM(
                SELECT City, Day, quartile ,AVG([Prezzo EUR/l]) as "MEDIA_GIORNO"
                FROM df_FEB_da_analizzare FEB
                JOIN df_car_fleet_FEB cff 
                ON FEB.[Plate number] = cff.Plate
                WHERE [Prod.] = "GASOLIO"
                GROUP BY City, Day
            )
            GROUP BY City,quartile
            ORDER BY City    
        )
        GROUP BY "Settimana"
    )
)
"""


#SUPER SENZA PB
query2="""
SELECT Settimane_al_mese, City, Settimana_minima
FROM(
    SELECT *,
    CASE 
        WHEN "Settimana" = "Q1" THEN "1"
        WHEN "Settimana" = "Q2" THEN "2"
        WHEN "Settimana" = "Q3" THEN "3"
        WHEN "Settimana" = "Q4" THEN "4"
            ELSE "Settimana"
        END AS "Settimane_al_mese"
    FROM(
        SELECT quartile as "Settimana" , City ,MIN(MEDIA_SETTIMANA) AS "Settimana_minima"
        FROM(
            SELECT City,quartile, avg("MEDIA_GIORNO") as "MEDIA_SETTIMANA"
            FROM(
                SELECT City, Day, quartile ,AVG([Prezzo EUR/l]) as "MEDIA_GIORNO"
                FROM df_FEB_da_analizzare FEB
                JOIN df_car_fleet_FEB cff 
                ON FEB.[Plate number] = cff.Plate
                WHERE [Prod.] = "SUPER SENZA PB"
                GROUP BY City, Day
            )
            GROUP BY City,quartile
            ORDER BY City    
        )
        GROUP BY "Settimana"
    )
)
"""


df_service_type=sqldf(query2,globals())
print(df_service_type)

In [ ]:
#calcolare quale macchina dura più km prima che ne riforniscano di benzina
km_FEB=[]
nose=[]
diff=[]
km_diff_media=pd.DataFrame()


query1="""
SELECT *
FROM df_FEB_da_analizzare FEB
JOIN df_car_fleet_FEB cff 
ON FEB.[Plate number] = cff.Plate
"""
df_FEB_km=sqldf(query1,globals())

#pulizia outlayer
for i in df_FEB_km["Plate number"].unique():
    wasa=df_FEB_km[df_FEB_km["Plate number"].isin([i])]
    if ((wasa["Plate number"].value_counts()).get(i,0)) > 1:
        q1=wasa["Km"].quantile(0.25)
        q3=wasa["Km"].quantile(0.75)
        iqr=q3-q1
        low=q1-1.5*iqr
        up=q3+1.5*iqr
        nose.append(wasa[(wasa["Km"] >= low) & (wasa["Km"] <= up)])
km_FEB=pd.concat(nose,ignore_index=True)


for i in km_FEB["Plate number"].unique():
    wasa=df_FEB_km[df_FEB_km["Plate number"].isin([i])]
    for j in range(len(wasa)):
        if j != max(range(len(wasa))):
            diff.append(int(abs((wasa.iloc[(j+1)]["Km"]) - (wasa.iloc[j]["Km"]))))

    km_diff=pd.DataFrame({
        "Kilometri":diff,
        "Plate":i
    })

    km_diff_media=pd.concat([km_diff_media,km_diff])
    
    


query1="""
SELECT Plate, CAST(AVG(Kilometri) AS INT) AS "Durata Km"
FROM km_diff_media
WHERE Kilometri < 3000
GROUP BY Plate
ORDER BY "Durata Km" DESC
LIMIT 10
"""

result_media_km=sqldf(query1,globals())


result=pd.merge(result_media_km,df_car_fleet_FEB, on="Plate")
result=result[["Modello","Marchio","Durata Km","Plate"]]
result=result.drop_duplicates(subset="Plate",ignore_index=True)


print(result)